# AoC 2024 Day 1 — Historian Hysteria

**Spark — window ranking + join**

Puzzle: <https://adventofcode.com/2024/day/1>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

---

## The puzzle

Two columns of location IDs, one per line.

- **Part 1** — pair the two lists up smallest-with-smallest, second-smallest-with-second-smallest, and so on, then sum the absolute difference of each pair.
- **Part 2** — for each value in the left list, multiply it by the number of times it appears in the right list, and sum those scores.

## The approach

This is the rare AoC day that is *naturally relational*, which makes it a good warm-up.

"Sort both sides and pair by rank" is a **window function**: `row_number()` over each column independently gives every value a rank, and joining the two ranked frames on `rank` reproduces the pairing without ever holding both sorted lists in memory at once.

"How often does it appear" is a **groupBy + join** — a frequency table joined back onto the left column. The `left` join matters: values that never appear on the right must score zero, not vanish.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day01

spark = get_spark('aoc-2024-day01')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = """\
3   4
4   3
2   5
1   3
3   9
3   3
"""

print('part 1:', day01.part1(spark, EXAMPLE), '(expected 11)')
print('part 2:', day01.part2(spark, EXAMPLE), '(expected 31)')

### Watch the pairing happen

The two ranked frames are what the join consumes. Notice each column is sorted *independently* — row 1 of `left` has nothing to do with row 1 of the input.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

df = day01.parse(spark, EXAMPLE)
df.show()

left = df.select(F.row_number().over(Window.orderBy('left_id')).alias('rank'), 'left_id')
right = df.select(F.row_number().over(Window.orderBy('right_id')).alias('rank'), 'right_id')

left.join(right, on='rank').orderBy('rank').withColumn(
    'distance', F.abs(F.col('left_id') - F.col('right_id'))
).show()

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 1)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

for part in (1, 2):
    fn = getattr(day01, f'part{part}')
    started = time.perf_counter()
    answer = fn(spark, data)
    print(f'part {part}: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Notes & gotchas

- The unpartitioned `Window.orderBy(...)` triggers Spark's *"No Partition Defined for Window operation!"* warning. It is correct here — the puzzle needs a **global** ordering, and a partitioned window would rank within groups instead. At 1000 rows the single-partition shuffle costs nothing; at a billion it would be the thing to redesign.
- Try swapping the `left` join in part 2 for an `inner` join and confirm the answer is unchanged *for this input* — then think about why that is luck, not correctness.